In [17]:
import json
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel,Field
from typing import Optional
import openai
import os

In [2]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:40019/v1"
)




def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,

        # stream=True  # 开启流式输出,
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""

user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input},]
qa_base(input)
    

'中国的首都是北京。'

### 第一步映射出相应疾病核保结论

In [60]:
home_path = "save"
path_names = os.listdir(home_path)
print(len(path_names))
# print(path_names)
select_one_path = [path for path in path_names if "tangniaobing" in path]
print(select_one_path)
one_path = os.path.join(home_path,select_one_path[1])
print(one_path)
list_file = os.listdir(one_path)
con_file = [file for file in list_file if "xlsx" in file]
file_tree = os.path.join(one_path,con_file[0])
print(file_tree)
df = pd.ExcelFile(file_tree)
print(df.sheet_names)
if "寿险" in df.sheet_names:
    print("ok")

def index_underwriting_conc(disease_name,type_insurance):
    home_path = "save"
    path_names = os.listdir(home_path)
    print(len(path_names))
    # print(path_names) 
    #检索相应疾病核保结论文件夹，检索方法需要优化
    select_one_path = [path for path in path_names if disease_name in path]
    #可能会有多个检索结果，
    one_path = os.path.join(home_path,select_one_path[1])
    list_file = os.listdir(one_path)
    con_file = [file for file in list_file if "xlsx" in file]
    file_tree = os.path.join(one_path,con_file[0])
    df = pd.ExcelFile(file_tree)
    print(df.sheet_names)
    if type_insurance in df.sheet_names:
        df_life = pd.read_excel(file_tree,sheet_name=type_insurance,keep_default_na=False)
    else:
        print(f"文件{file_tree}中没有险种：{type_insurance}")
        df_life = None
    return df_life

disease_name = "2xingtangniaobing"
type_insurance = "寿险"
df_life = index_underwriting_conc(disease_name,type_insurance)



376
['renshenqijiandetangniaobing', '2xingtangniaobing', '1xingtangniaobing', '2xingtangniaobing的副本']
save/2xingtangniaobing
save/2xingtangniaobing/2xingtangniaobing.xlsx
['寿险', '意外险种', '重疾和癌症', '失能收入', '豁免保费和永久完全性失能', '住院收入保障', '长期护理险']
ok
376
['寿险', '意外险种', '重疾和癌症', '失能收入', '豁免保费和永久完全性失能', '住院收入保障', '长期护理险']


In [56]:
df_life.head()

,1_级问题,2_级问题,3_级问题,4_级问题,寿险
0,病程小于6个月,,,,延期
1,病程为6个月－1年,<15岁,,,延期
2,病程为6个月－1年,15-19岁,HbA1c < 7%,,+200
3,病程为6个月－1年,15-19岁,HbA1c ≥ 7%,,拒保
4,病程为6个月－1年,20-39岁,HbA1c < 7%,,+150


In [47]:
df_life[(df_life["1_级问题"]=="病程为6个月－1年") & (df_life["2_级问题"]=="<15岁")]

,1_级问题,2_级问题,3_级问题,4_级问题,寿险
1,病程为6个月－1年,<15岁,,,延期


In [62]:
def index_underwriting_conc(disease_name,type_insurance):
    home_path = "save"
    path_names = os.listdir(home_path)
    print(len(path_names))
    # print(path_names) 
    #检索相应疾病核保结论文件夹，检索方法需要优化
    select_one_path = [path for path in path_names if disease_name in path]
    #可能会有多个检索结果，
    one_path = os.path.join(home_path,select_one_path[1])
    list_file = os.listdir(one_path)
    con_file = [file for file in list_file if "xlsx" in file]
    file_tree = os.path.join(one_path,con_file[0])
    df = pd.ExcelFile(file_tree)
    print(df.sheet_names)
    if type_insurance in df.sheet_names:
        df_life = pd.read_excel(file_tree,sheet_name=type_insurance,keep_default_na=False)
    else:
        print(f"文件{file_tree}中没有险种：{type_insurance}")
        df_life = None
    return df_life


def conclusion_gen(input_data,df_life):
    historys_list = []
    message = {"role": "user", "content": f"你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:{input_data}"}
    historys_list.append(message)
    question_prompt = """
        请判断该病人是否有如下的情况，情况列表：{node_questions},
        注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
        注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
        结果示例：分析理由#选项值
        """
    #如果无法更具给定的数据判断则返回其他
    all_columns = df_life.columns
    columns_conc = all_columns[-1]
    run_flag = True
    level_num = 1
    end_conc = ""
    while run_flag:
        print("level num:",level_num)
        node_questions = []
        question_map_level = {}
        columns_name = f"{level_num}_级问题"
        if len(df_life)==0:
            print("无下一级问题,或中间轮回答answer_key有误")
            break
        if columns_name not in all_columns:
            print("问答结束")
            break
        node_questions = list(set(df_life[columns_name].tolist()))
        print(node_questions)
        question_prompt_input = question_prompt.format(node_questions=node_questions)
        print(question_prompt_input)
        message = {"role": "user", "content": question_prompt_input}
        historys_list.append(message)
        messages = historys_list
        # print(messages)
        answer = qa_base(messages)
        historys_list.append({"role": "assistant", "content": answer})
        print("answer:",answer)
        level_num += 1

        answer_key = answer.split("#")[-1]
        print("answer_key",answer_key)
        df_life = df_life[df_life[columns_name]==answer_key]
        #判断是否能得出最终结论
        if len(df_life)==1:
            end_conc = list(df_life[columns_conc].tolist())[0]
            print("获得结论,问答结束")
            break
    return end_conc,historys_list




In [64]:
print("第一步：\n 根据疾病名称和险种获得相应核保结论知识")
disease_name = "2xingtangniaobing"
type_insurance = "寿险"
df_life = index_underwriting_conc(disease_name,type_insurance)

print("第二步: \n 大模型多轮问答获得结论")
# input_data = "" #case 疾病信息
input_data = "年龄：71岁\
                临床诊断化验项：_GLU（空腹血糖）:8.82_ALP（碱性磷酸酶）:63.4_GGT（谷氨酰转肽酶）:32.2_TBIL（总胆红素）:5.84_DBIL（直接胆红素）:1.96_\
                AST（谷草转氨酶）:19.70_ALT（谷丙转氨酶）:18.30_TG（甘油三脂）:1.47_BUN（血尿素氮）:5.86_UA（尿酸）:307.50_CR(肌酐):43.60_糖化血红蛋白:9.12_\
                    HBSAG（乙肝表面抗原）:阴性(-)_HDL（高密度脂蛋白）:1.24_LDL（低密度脂蛋白）:2.29_糖尿病"
first_end_result , historys_list = conclusion_gen(input_data,df_life)
print("结论：",first_end_result)
print(historys_list)

第一步：
 根据疾病名称和险种获得相应核保结论知识
376
['寿险', '意外险种', '重疾和癌症', '失能收入', '豁免保费和永久完全性失能', '住院收入保障', '长期护理险']
第二步: 
 大模型多轮问答获得结论
level num: 1
['病程>15年', '病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '存在并发症和/或其他风险因素']

        请判断该病人是否有如下的情况，情况列表：['病程>15年', '病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '存在并发症和/或其他风险因素'],
        注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
        注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
        结果示例：分析理由#选项值
        


answer: 该病人的空腹血糖为8.82 mmol/L，糖化血红蛋白为9.12%，这些指标均高于正常范围，提示该病人患有糖尿病。根据提供的信息，虽然没有明确指出病人的糖尿病病程，但高血糖和高糖化血红蛋白水平表明其糖尿病病程可能较长。同时，该病人71岁的年龄也增加了长期患有糖尿病的可能性。因此，根据这些信息，最有可能的情况是该病人病程超过15年。

分析理由#病程>15年
answer_key 病程>15年
level num: 2
['15-19岁', '≥50岁', '40-49岁', '20-39岁']

        请判断该病人是否有如下的情况，情况列表：['15-19岁', '≥50岁', '40-49岁', '20-39岁'],
        注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
        注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
        结果示例：分析理由#选项值
        
answer: 根据提供的病人基本信息，该病人的年龄为71岁，因此符合“≥50岁”这一分类。

分析理由#≥50岁
answer_key ≥50岁
level num: 3
['HbA1c < 7%', 'HbA1c 7.1-8%', 'HbA1c 8.1-10%', 'HbA1c > 10%']

        请判断该病人是否有如下的情况，情况列表：['HbA1c < 7%', 'HbA1c 7.1-8%', 'HbA1c 8.1-10%', 'HbA1c > 10%'],
        注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
        注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
        结果示例：分析理由#选项值
        
answer: 根据提供的信息，病人的糖化血红蛋白（HbA1c）水平为9.12%，这落在了8.1%到10%的范围内。

分析理由#HbA1c 8.1-10%
answer_key HbA1c 8.1-10%
获得结论,问答结束
结论： +175
[{'role': 'user', 'content': '你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:年龄